# KernelForge on a Colab T4

Builds the CUDA kernels, checks them against the NumPy reference, benchmarks them,
and profiles v3. Run the cells top to bottom.

**Before anything else: Runtime > Change runtime type > T4 GPU.** Without a GPU the
tests will skip rather than fail, which looks like success and is not.

Nothing from this notebook counts until cell 3 shows GPU tests *running* (not `s`)
and cell 4 writes real rows to `bench/results.csv`.


## 0. Confirm the GPU


In [ ]:
!nvidia-smi
import torch
assert torch.cuda.is_available(), 'No GPU: Runtime > Change runtime type > T4 GPU'
cap = torch.cuda.get_device_capability()
ARCH = f'sm_{cap[0]}{cap[1]}'
print(torch.cuda.get_device_name(0), ARCH)


## 1. Get the code

Two ways, pick one. If the repo is public on GitHub, clone it. Otherwise zip the
`kernelforge-cuda` folder on your Mac, run the upload cell, and choose the zip.


In [ ]:
# (a) clone - only works once the repo is public
!git clone -q https://github.com/mghadia1/kernelforge-cuda 2>/dev/null && echo cloned || echo 'not on GitHub yet - use the upload cell below'


In [ ]:
# (b) upload a zip of the project folder instead
import os, zipfile, pathlib
if not pathlib.Path('kernelforge-cuda').exists():
    from google.colab import files
    up = files.upload()          # choose kernelforge-cuda.zip
    name = next(iter(up))
    with zipfile.ZipFile(name) as z:
        z.extractall('.')
    print(sorted(os.listdir('kernelforge-cuda')))


In [ ]:
%cd kernelforge-cuda
!ls


## 2. Build

One shared library holds every version, so the benchmark switches between them by
symbol name through ctypes.


In [ ]:
!pip install -q -e '.[dev]'
!make ARCH=$ARCH 2>&1 | tail -20
!ls -la build/


## 3. Correctness gate

Every kernel must match the NumPy reference within 1e-4 **and** return the same
indices. Look at the output: you want `passed`, and you want the count of skipped
tests to be 0. Skips here mean the library did not load.


In [ ]:
!python -m pytest -v 2>&1 | tail -45


## 4. Benchmark sweep

15 timed repeats, first discarded, 3 warmup runs, median and p95, every
implementation verified against the reference before it is timed. The 1M row takes
a few minutes, mostly in the CPU baseline.


In [ ]:
!python bench/run.py --out bench/results.csv --repeats 15 2>&1 | tail -60


## 5. Turn the CSV into the tables RESULTS.md wants

Paste the printed markdown straight into `bench/RESULTS.md`. Do not retype numbers
by hand - that is how a benchmark quietly becomes fiction.


In [ ]:
!pip install -q tabulate
import pandas as pd
df = pd.read_csv('bench/results.csv')
print('device:', df.device.iloc[0])
print('all implementations matched the reference:', bool(df.indices_match.all()),
      '| worst abs err:', df.max_abs_err.max())

for b in sorted(df.B.unique()):
    sub = df[df.B == b]
    piv = sub.pivot_table(index='N', columns='impl', values='median_ms')
    order = [c for c in ['cpu_numpy','v0_naive','v1_shared','v2_warp','v3_topk','cublas','torch_gpu'] if c in piv.columns]
    print(f'\n### End-to-end latency, median ms (B = {b})\n')
    print(piv[order].round(3).to_markdown())
    if 'cpu_numpy' in piv.columns:
        spd = piv[order].div(piv['cpu_numpy'], axis=0)
        print(f'\n### Speedup over cpu_numpy (x, B = {b})\n')
        print(spd.drop(columns=['cpu_numpy']).round(1).to_markdown())


## 6. Where the time actually goes

This is the v3 argument in one table: v0-v2 ship a B x N score matrix back over PCIe
to extract B x k numbers. If the device-to-host row is not the dominant cost for v2
at N = 1M, then v3 is solving a problem you do not have - say so in RESULTS.md.


In [ ]:
import sys; sys.path.insert(0, 'src')
import reference, runner
q, X = reference.make_data(1_000_000, 32, seed=0)
for v in ('v2_warp', 'v3_topk'):
    runner.run(v, q, X, 5)                      # warm up, then measure
    _, _, t = runner.run(v, q, X, 5)
    print(f'{v:<9} h2d {t.h2d_ms:8.2f} | kernel {t.kernel_ms:8.2f} | '
          f'd2h {t.d2h_ms:8.2f} | host top-k {t.host_topk_ms:8.2f} | total {t.total_ms:8.2f} ms')


## 7. Nsight Compute profile of v3

Colab permits the basic sections. The two numbers to write down: achieved occupancy
and DRAM throughput as a percentage of peak. The prediction in RESULTS.md is that
this kernel is memory-bound at roughly 0.5 FLOP/byte, well below the T4's ridge point
near 25 - so DRAM throughput should be high and SM throughput low. If it comes back
the other way, the prediction was wrong and RESULTS.md says that.


In [ ]:
!which ncu || ls /usr/local/cuda/bin/ncu || echo 'ncu not installed on this runtime'
!ncu --set basic --target-processes all \
     python bench/run.py --profile-once --n 100000 --b 32 2>&1 | tail -40


## 8. Save the evidence

Download `results.csv` and commit it next to the numbers you paste into RESULTS.md,
so every figure in that file has a machine-written source.


In [ ]:
from google.colab import files
files.download('bench/results.csv')


---

### After the run

The repo flips to `resume_eligible: yes` only when all five hold:

1. `make` succeeded here;
2. pytest ran the GPU tests and passed with **0 skips**;
3. `bench/results.csv` has real rows and RESULTS.md quotes them;
4. the Nsight numbers are recorded and interpreted;
5. you can explain one design choice and one failure mode unaided.

Good candidates for #5: *why the v1 tile row is padded to 33 floats*, and *what
happens to v3 when k exceeds 8*.
